# Image Classifier 
## Training deep neural network with less data using bottleneck features

Importing necessary libraries

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import os
import h5py
from keras.preprocessing.image import ImageDataGenerator
from keras import optimizers
from keras.models import Sequential
from keras.layers import Convolution2D, MaxPooling2D, ZeroPadding2D
from keras.layers import Activation, Dropout, Flatten, Dense

# Forcing it to use theono as the backend
from keras import backend as K
K.set_image_dim_ordering('th')

In [8]:
# path to the model weights files.
weights_path = 'data/vgg16_weights.h5'
top_model_weights_path = 'data/bottleneck_fc_model.h5'
# dimensions of our images.
img_width, img_height = 150, 150

train_data_dir = 'data/catsdogs/train'
validation_data_dir = 'data/catsdogs/validation'

In [9]:
nb_train_samples = 2000
nb_validation_samples = 800
nb_epoch = 5 # Originally 50

In [10]:
datagen = ImageDataGenerator(rescale=1./255)

In [11]:
model = Sequential()
model.add(ZeroPadding2D((1, 1), input_shape=(3, img_width, img_height)))

model.add(Convolution2D(64, 3, 3, activation='relu', name='conv1_1'))
model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(64, 3, 3, activation='relu', name='conv1_2'))
model.add(MaxPooling2D((2, 2), strides=(2, 2)))

model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(128, 3, 3, activation='relu', name='conv2_1'))
model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(128, 3, 3, activation='relu', name='conv2_2'))
model.add(MaxPooling2D((2, 2), strides=(2, 2)))

model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(256, 3, 3, activation='relu', name='conv3_1'))
model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(256, 3, 3, activation='relu', name='conv3_2'))
model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(256, 3, 3, activation='relu', name='conv3_3'))
model.add(MaxPooling2D((2, 2), strides=(2, 2)))

model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(512, 3, 3, activation='relu', name='conv4_1'))
model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(512, 3, 3, activation='relu', name='conv4_2'))
model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(512, 3, 3, activation='relu', name='conv4_3'))
model.add(MaxPooling2D((2, 2), strides=(2, 2)))

model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(512, 3, 3, activation='relu', name='conv5_1'))
model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(512, 3, 3, activation='relu', name='conv5_2'))
model.add(ZeroPadding2D((1, 1)))
model.add(Convolution2D(512, 3, 3, activation='relu', name='conv5_3'))
model.add(MaxPooling2D((2, 2), strides=(2, 2)))

In [12]:
assert os.path.exists(weights_path), 'Model weights not found (see "weights_path" variable in script).'
f = h5py.File(weights_path)
for k in range(f.attrs['nb_layers']):
    if k >= len(model.layers):
        # we don't look at the last (fully-connected) layers in the savefile
        break
    g = f['layer_{}'.format(k)]
    weights = [g['param_{}'.format(p)] for p in range(g.attrs['nb_params'])]
    model.layers[k].set_weights(weights)
f.close()

In [13]:
generator = datagen.flow_from_directory(
        train_data_dir,
        target_size=(img_width, img_height),
        batch_size=16,
        class_mode=None,
        shuffle=False)
bottleneck_features_train = model.predict_generator(generator, nb_train_samples)
np.save(open('data/catsdogs/savedmodels/bottleneck_features_train.npy', 'w'), bottleneck_features_train)

Found 2222 images belonging to 2 classes.


In [14]:
generator = datagen.flow_from_directory(
        validation_data_dir,
        target_size=(img_width, img_height),
        batch_size=16,
        class_mode=None,
        shuffle=False)
bottleneck_features_validation = model.predict_generator(generator, nb_validation_samples)
np.save(open('data/catsdogs/savedmodels/bottleneck_features_validation.npy', 'w'), bottleneck_features_validation)

Found 1222 images belonging to 2 classes.


In [16]:
train_data = np.load(open('data/catsdogs/savedmodels/bottleneck_features_train.npy'))
train_labels = np.array([0] * (nb_train_samples / 2) + [1] * (nb_train_samples / 2))

validation_data = np.load(open('data/catsdogs/savedmodels/bottleneck_features_validation.npy'))
validation_labels = np.array([0] * (nb_validation_samples / 2) + [1] * (nb_validation_samples / 2))

In [17]:
top_model = Sequential()
top_model.add(Flatten(input_shape=train_data.shape[1:]))
top_model.add(Dense(256, activation='relu'))
top_model.add(Dropout(0.5))
top_model.add(Dense(1, activation='sigmoid'))

top_model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
top_model.fit(train_data, train_labels,
          nb_epoch=nb_epoch, batch_size=160,
          validation_data=(validation_data, validation_labels))

Train on 2000 samples, validate on 800 samples
Epoch 1/5
2000/2000 [==============================] - 0s - loss: 2.2792 - acc: 0.5715 - val_loss: 0.7415 - val_acc: 0.6150
Epoch 2/5
2000/2000 [==============================] - 0s - loss: 0.5984 - acc: 0.7345 - val_loss: 0.6651 - val_acc: 0.6163
Epoch 3/5
2000/2000 [==============================] - 0s - loss: 0.4571 - acc: 0.7875 - val_loss: 0.7389 - val_acc: 0.6363
Epoch 4/5
2000/2000 [==============================] - 0s - loss: 0.4161 - acc: 0.8155 - val_loss: 0.6835 - val_acc: 0.6225
Epoch 5/5
2000/2000 [==============================] - 0s - loss: 0.4454 - acc: 0.7855 - val_loss: 0.7083 - val_acc: 0.6750


In [18]:
model.add(top_model)
for layer in model.layers[:25]:
    layer.trainable = False

In [19]:
model.compile(loss='binary_crossentropy',
              optimizer=optimizers.SGD(lr=1e-4, momentum=0.9),
              metrics=['accuracy'])

train_datagen = ImageDataGenerator(
        rescale=1./255,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [20]:
train_generator = train_datagen.flow_from_directory(
        train_data_dir,
        target_size=(img_height, img_width),
        batch_size=320,
        class_mode='binary')

Found 2222 images belonging to 2 classes.


In [21]:
validation_generator = test_datagen.flow_from_directory(
        validation_data_dir,
        target_size=(img_height, img_width),
        batch_size=320,
        class_mode='binary')

Found 1222 images belonging to 2 classes.


In [22]:
model.fit_generator(
        train_generator,
        samples_per_epoch=nb_train_samples,
        nb_epoch=nb_epoch,
        validation_data=validation_generator,
        nb_val_samples=nb_validation_samples)

Epoch 1/5
2222/2000 [=================================] - 468s - loss: 0.3541 - acc: 0.8443 - val_loss: 0.3037 - val_acc: 0.8781
Epoch 2/5
2222/2000 [=================================] - 472s - loss: 0.3324 - acc: 0.8717 - val_loss: 0.3136 - val_acc: 0.8714
Epoch 3/5
1920/2000 [===========================>..] - ETA: 12s - loss: 0.3377 - acc: 0.8635

/Users/mohammed/anaconda/lib/python2.7/site-packages/keras/engine/training.py:1462: UserWarning: Epoch comprised more than `samples_per_epoch` samples, which might affect learning results. Set `samples_per_epoch` correctly to avoid this warning.
  warnings.warn('Epoch comprised more than '


KeyboardInterrupt: 